# Fusion: audio + RF (vision plugs in later)

**Inputs (from your two finished notebooks):** `rf_fusion_export.npz` (RF) and `audio_fusion_final_pretrained_ext.npz` (audio).
**What this notebook does**
1. Joins RF frames to audio windows on *(split, class, recording, clip, time)* and prints a match report.
2. Calibrates each branch, then runs baselines (average / max / min) and **Method A: stacking** (logistic regression on calibrated logits, fitted on **val**).
3. **Method B: GMU** (gated multimodal unit) on the frozen embeddings, trained on **train**, early-stopped on **val**.
4. Tests everything on **test**, including a sensor that is missing or sends garbage.

Everything uses TRIDENT's *official* split (the same one both branches were trained on). Recordings are shared between the official splits, so absolute numbers are optimistic. Say so in the report.

In [ ]:
# ============================================================
# 1. SETUP  (edit only the block marked EDIT)
# ============================================================
import os, re, json, glob
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# ---------------- EDIT ----------------
BASE         = "/content/drive/MyDrive"
RF_EXPORT    = f"{BASE}/RF_Models_v2/rf_fusion_export.npz"
AUDIO_EXPORT = f"{BASE}/audio_fusion_final_pretrained_ext.npz"
OUT_DIR      = f"{BASE}/Fusion_v1"
VISION_CSV   = None      # later: csv with columns split, cls, recording, clip, frame, p_v (+ optional v_bird, v_airplane)
SEED         = 42
# --------------------------------------

def locate(path):
    if os.path.exists(path): return path
    hits = glob.glob(f"{BASE}/**/{os.path.basename(path)}", recursive=True)
    if hits:
        print("not at the expected path, using:", hits[0]); return hits[0]
    raise FileNotFoundError(f"{path} not found. Is Drive mounted from the account that holds the files?")

RF_EXPORT, AUDIO_EXPORT = locate(RF_EXPORT), locate(AUDIO_EXPORT)
os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.RandomState(SEED)
print("RF   :", RF_EXPORT); print("audio:", AUDIO_EXPORT); print("out  :", OUT_DIR)

In [ ]:
# ============================================================
# 2. LOAD RF  (P(drone) and 2048-d embedding per image; 1 = Drone)
# ============================================================
z = np.load(RF_EXPORT, allow_pickle=True)
print("RF export meta:", z["meta"])
E_r_all = z["emb"].astype(np.float32)

rf = pd.DataFrame(dict(
    split=z["split_official"].astype(str), recording=z["recording"].astype(str),
    filename=z["filename"].astype(str), y=z["y_drone"].astype(int), p_r=z["p_drone"].astype(np.float64)))
rf["r_row"] = np.arange(len(rf))                         # row in E_r_all
rf["cls"]    = np.where(rf.y == 1, "Drone", "Background")
rf["recnum"] = rf.recording.map(lambda s: int(re.findall(r"\d+", s)[-1]))

pat = re.compile(r"^(\d+)_frame_(\d+)\.(?:jpe?g|png|bmp|webp)$", re.I)
parsed = rf.filename.map(lambda s: pat.match(s))
bad = parsed.isna()
print(f"RF images whose filename does not parse (dropped): {bad.sum()}  {rf.filename[bad].tolist()[:5]}")
rf = rf[~bad].copy()
rf["clip"]  = parsed[~bad].map(lambda m: int(m.group(1)))
rf["frame"] = parsed[~bad].map(lambda m: int(m.group(2)))
print(rf.groupby("split").size().to_dict(), "| frames per clip:", rf.groupby(["split","cls","recnum","clip"]).size().value_counts().to_dict())

In [ ]:
# ============================================================
# 3. LOAD AUDIO  (P(drone) = column 1, 512-d embedding per 1 s window)
#    window_meta rows = (grouped_split, ORIGINAL split, recording, segment, start_sec)
#    NOTE: the npz key "split" is the GROUPED split -> we deliberately use the original one from window_meta.
# ============================================================
a = np.load(AUDIO_EXPORT, allow_pickle=True)
print("audio keys:", a.files)
Pa, E_a_all, meta, ya = a["window_probs"], a["window_emb"].astype(np.float32), a["window_meta"], a["window_y"]
au = pd.DataFrame(dict(
    split=[str(m[1]) for m in meta], recording=[str(m[2]) for m in meta],
    clip=[int(m[3]) for m in meta], start=[float(m[4]) for m in meta],
    y_a=np.asarray(ya).astype(int), p_a=Pa[:, 1].astype(np.float64)))
au["a_row"]  = np.arange(len(au))
au["cls"]    = np.where(au.y_a == 1, "Drone", "Background")
au["recnum"] = au.recording.map(lambda s: int(re.findall(r"\d+", s)[-1]))
au["widx"]   = np.rint(au.start / 0.5).astype(int)        # 0..18  (1 s windows, 0.5 s hop, 10 s clips)
KEYS = ["split", "cls", "recnum", "clip", "widx"]
assert au.duplicated(KEYS).sum() == 0, "audio keys are not unique"
print(au.groupby("split").size().to_dict(), "| windows per clip:", au.groupby(["split","cls","recnum","clip"]).size().value_counts().to_dict())

In [ ]:
# ============================================================
# 4. JOIN: RF frame k (0.25 s steps) -> the audio window whose centre is closest in time
#    window j covers [0.5 j, 0.5 j + 1) s.   frame k starts at 0.25 (k-1) s.   j = clip(round(0.5 (k-1) - 0.75), 0, 18)
#    (Labels are constant inside a clip, so even a wrong within-clip alignment cannot flip a label.)
# ============================================================
rf["widx"] = np.clip(np.rint(0.5 * (rf.frame - 1) - 0.75), 0, 18).astype(int)
M = rf.merge(au[KEYS + ["p_a", "y_a", "a_row"]], on=KEYS, how="left")
M["matched"] = M.a_row.notna()

print("=== MATCH REPORT (RF frames that found an audio window) ===")
rep = M.groupby("split").agg(rf_frames=("matched", "size"), matched=("matched", "sum"))
rep["pct"] = (100 * rep.matched / rep.rf_frames).round(1)
print(rep)
um = M[~M.matched].groupby(["split", "cls", "recnum", "clip"]).size()
print(f"\nRF clips with NO audio partner: {len(um)}")
print(um.head(12).to_string() if len(um) else "")

F = M[M.matched].copy().reset_index(drop=True)
F["a_row"] = F.a_row.astype(int)
bad_lab = (F.y != F.y_a).sum()
print(f"\nlabel disagreements between the RF and audio side of the join: {bad_lab}  (must be 0)")
assert bad_lab == 0, "join is wrong: labels disagree"
print("fused rows:", F.groupby(["split", "cls"]).size().unstack())

In [ ]:
# ============================================================
# 5. (optional) VISION: plug a per-frame vision score into the same table
#    csv columns: split, cls, recording, clip, frame, p_v   (optional: v_bird, v_airplane)
# ============================================================
MODS = ["a", "r"]
if VISION_CSV:
    v = pd.read_csv(VISION_CSV)
    v["recnum"] = v.recording.map(lambda s: int(re.findall(r"\d+", str(s))[-1]))
    vcols = [c for c in ("p_v", "v_bird", "v_airplane") if c in v.columns]
    F = F.merge(v[["split", "cls", "recnum", "clip", "frame"] + vcols], on=["split", "cls", "recnum", "clip", "frame"], how="inner").reset_index(drop=True)
    MODS = ["a", "r", "v"]
    print("rows with vision:", len(F))

E_raw = {"a": E_a_all[F.a_row.values], "r": E_r_all[F.r_row.values]}
if "v" in MODS:
    E_raw["v"] = F[vcols].values.astype(np.float32)
y  = F.y.values.astype(int)
tr = np.where(F.split == "Train")[0]; va = np.where(F.split == "Validation")[0]; te = np.where(F.split == "Test")[0]
print("modalities:", MODS, "| train/val/test rows:", len(tr), len(va), len(te))

In [ ]:
# ============================================================
# 6. CALIBRATION, BASELINES, METHOD A (stacking)
# ============================================================
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score

def logit(p): p = np.clip(p, 1e-4, 1 - 1e-4); return np.log(p / (1 - p))
def sig(x):   return 1 / (1 + np.exp(-x))

def fit_T(l, yy):                                   # temperature scaling on val
    nll = lambda lt: np.mean(np.logaddexp(0, l / np.exp(lt)) - yy * (l / np.exp(lt)))
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))

def metrics(yy, p, thr):
    yh = (p >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(yy, yh, labels=[0, 1]).ravel()
    return dict(acc=accuracy_score(yy, yh), macro_f1=f1_score(yy, yh, average="macro"),
                miss=fn / max(fn + tp, 1), false_alarm=fp / max(fp + tn, 1),
                auc=roc_auc_score(yy, p) if len(set(yy)) > 1 else np.nan)

def best_thr(yy, p):                                # chosen on VAL only
    g = np.arange(0.05, 0.96, 0.01); return float(g[np.argmax([f1_score(yy, (p >= t).astype(int), average="macro") for t in g])])

P_raw = {m: F[f"p_{m}"].values for m in MODS if f"p_{m}" in F}
if "v" in MODS: P_raw["v"] = F["p_v"].values
# Vision is zero-inflated (the detector says "nothing" on most frames, drones included), so a plain logit would treat
# "no detection" as strong evidence of background. Instead vision gets a log-likelihood ratio per score bin, fitted on val:
# a detection is strong evidence FOR drone, "no detection" is nearly neutral, and 0 means "no information" (= sensor missing).
V_EDGES = (0.0, 0.25, 0.5)
def vbin(p): return np.select([p <= V_EDGES[0], p <= V_EDGES[1], p <= V_EDGES[2]], [0, 1, 2], 3)
VLLR = None
if "v" in MODS:
    bv, n1, n0 = vbin(P_raw["v"][va]), (y[va] == 1).sum(), (y[va] == 0).sum()
    c1 = np.array([((bv == b) & (y[va] == 1)).sum() for b in range(4)]) + 1
    c0 = np.array([((bv == b) & (y[va] == 0)).sum() for b in range(4)]) + 1
    VLLR = np.log(c1 / (n1 + 4)) - np.log(c0 / (n0 + 4))
    print("vision bins [none | 0-.25 | .25-.5 | >.5]  drone seg:", c1 - 1, " bg seg:", c0 - 1, " LLR:", VLLR.round(2))
TEMP = {m: (1.0 if m == "v" else fit_T(logit(P_raw[m][va]), y[va])) for m in MODS}
Z = {m: (VLLR[vbin(P_raw[m])] if m == "v" else logit(P_raw[m]) / TEMP[m]) for m in MODS}    # calibrated evidence (logit scale), all rows
print("temperatures:", {m: round(t, 2) for m, t in TEMP.items()})

Zva = np.stack([Z[m][va] for m in MODS], 1)
LRS = {"stack": LogisticRegression(C=1.0, max_iter=1000).fit(Zva, y[va])}
aug, ya_ = [Zva], [y[va]]
for i in range(len(MODS)):                                     # stack_drop: also learn from "this sensor is missing" examples
    Zd = Zva.copy(); Zd[:, i] = 0; aug.append(Zd); ya_.append(y[va])
LRS["stack_drop"] = LogisticRegression(C=1.0, max_iter=1000).fit(np.vstack(aug), np.concatenate(ya_))
print("stack coefficients (higher = trusted more):", dict(zip(MODS, LRS["stack"].coef_[0].round(2))), "bias", LRS["stack"].intercept_.round(2))

def score_classic(Zs, mask, singles=False):
    Ps = np.stack([sig(Zs[m]) for m in MODS], 1); Mk = mask.astype(float)
    out = {"avg": (Ps * Mk).sum(1) / Mk.sum(1), "max": np.where(mask, Ps, -1).max(1), "min": np.where(mask, Ps, 2).min(1)}
    Zm = np.stack([np.where(mask[:, i], Zs[m], 0.0) for i, m in enumerate(MODS)], 1)
    for name, lr in LRS.items(): out[name] = lr.predict_proba(Zm)[:, 1]
    if singles:
        for i, m in enumerate(MODS): out[f"only_{m}"] = Ps[:, i]
    return out

full = np.ones((len(F), len(MODS)), bool)
Zall = {m: Z[m] for m in MODS}
THR = {}
for name, p in score_classic(Zall, full, singles=True).items():
    THR[name] = 0.5 if name.startswith("stack") else best_thr(y[va], p[va])

In [ ]:
# ============================================================
# 7. METHOD B: GMU on the frozen embeddings (needs torch; skipped automatically without it)
#    trained on TRAIN, early-stopped on VAL, evaluated on TEST. 5 seeds -> mean +/- std, and a 5-seed ensemble.
# ============================================================
try:
    import torch, torch.nn as nn
    HAVE_TORCH = True
except Exception:
    HAVE_TORCH = False
    print("torch not available -> GMU skipped")

mu = {m: E_raw[m][tr].mean(0) for m in MODS}; sd = {m: E_raw[m][tr].std(0) + 1e-6 for m in MODS}
Es = {m: ((E_raw[m] - mu[m]) / sd[m]).astype(np.float32) for m in MODS}      # standardised with TRAIN statistics

GMUS = []
if HAVE_TORCH:
    DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    class GMU(nn.Module):
        def __init__(s, dims, h=64, p=0.3):
            super().__init__()
            s.proj = nn.ModuleList([nn.Sequential(nn.Dropout(p), nn.Linear(d, h), nn.Tanh()) for d in dims])
            s.gate = nn.Linear(sum(dims), len(dims))
            s.out  = nn.Sequential(nn.Dropout(p), nn.Linear(h, 1))
        def forward(s, xs, mask):                     # xs: list of (B,d); mask (B,M) bool, True = sensor present
            xs = [x * mask[:, i:i+1].float() for i, x in enumerate(xs)]
            hs = torch.stack([pr(x) for pr, x in zip(s.proj, xs)], 1)               # B,M,h
            g  = s.gate(torch.cat(xs, -1)).masked_fill(~mask, -1e4)                  # absent sensors get no weight
            return s.out((torch.softmax(g, -1).unsqueeze(-1) * hs).sum(1)).squeeze(-1)

    T_ = {m: torch.from_numpy(Es[m]) for m in MODS}
    DIMS = [Es[m].shape[1] for m in MODS]

    @torch.no_grad()
    def gmu_prob(models, Er, mask):                   # Er: dict mod -> (n,d) float32 numpy;  mask: (n,M) bool numpy
        P = []
        for mdl in models:
            mdl.eval(); out = []
            for s in range(0, len(mask), 512):
                xs = [torch.from_numpy(Er[m][s:s+512]).to(DEV) for m in MODS]
                out.append(torch.sigmoid(mdl(xs, torch.from_numpy(mask[s:s+512]).to(DEV))).cpu().numpy())
            P.append(np.concatenate(out))
        return np.mean(P, 0)

    def train_gmu(seed, epochs=60, patience=8, bs=128, p_miss=0.2, noise=0.1):
        torch.manual_seed(seed); g = np.random.RandomState(seed)
        mdl = GMU(DIMS).to(DEV); opt = torch.optim.AdamW(mdl.parameters(), 1e-3, weight_decay=1e-2)
        yt = torch.from_numpy(y.astype(np.float32)); best, best_state, bad = 1e9, None, 0
        Er_va = {m: Es[m][va] for m in MODS}; ones_va = np.ones((len(va), len(MODS)), bool)
        for ep in range(epochs):
            mdl.train(); perm = g.permutation(tr)
            for s in range(0, len(perm), bs):
                b = perm[s:s+bs]
                if len(b) < 2: continue
                xs = [T_[m][b].to(DEV) + noise * torch.randn(len(b), Es[m].shape[1], device=DEV) for m in MODS]
                mask = torch.rand(len(b), len(MODS), device=DEV) > p_miss                     # modality dropout
                force = nn.functional.one_hot(torch.randint(0, len(MODS), (len(b),), device=DEV), len(MODS)).bool()
                mask = mask | (force & ~mask.any(1, keepdim=True))                            # never drop every sensor
                loss = nn.functional.binary_cross_entropy_with_logits(mdl(xs, mask), yt[b].to(DEV))
                opt.zero_grad(); loss.backward(); opt.step()
            pv = np.clip(gmu_prob([mdl], Er_va, ones_va), 1e-6, 1 - 1e-6)
            vl = float(-np.mean(y[va] * np.log(pv) + (1 - y[va]) * np.log(1 - pv)))
            if vl < best: best, bad, best_state = vl, 0, {k: v.detach().clone() for k, v in mdl.state_dict().items()}
            else:
                bad += 1
                if bad >= patience: break
        mdl.load_state_dict(best_state); return mdl, best, ep + 1

    for sd_ in range(5):
        mdl, vl, n_ep = train_gmu(SEED + sd_); GMUS.append(mdl); print(f"GMU seed {sd_}: best val loss {vl:.4f} after {n_ep} epochs")

    ones = np.ones((len(F), len(MODS)), bool)
    p_all = gmu_prob(GMUS, Es, ones); THR["gmu"] = best_thr(y[va], p_all[va])
    per_seed = [metrics(y[te], gmu_prob([m_], {k: v[te] for k, v in Es.items()}, ones[te]), THR["gmu"])["acc"] for m_ in GMUS]
    print(f"GMU test accuracy per seed: mean {np.mean(per_seed):.4f} +/- {np.std(per_seed):.4f}")

In [ ]:
# ============================================================
# 8. RESULTS: clean test, then a sensor that is missing / sends garbage
# ============================================================
def all_scores(Zs, Er, mask, singles=False):
    out = score_classic(Zs, mask, singles)
    if GMUS: out["gmu"] = gmu_prob(GMUS, Er, mask)
    return out

Zte = {m: Z[m][te] for m in MODS}; Ete = {m: Es[m][te] for m in MODS}; ok = np.ones((len(te), len(MODS)), bool)
clean = all_scores(Zte, Ete, ok, singles=True)
tab = pd.DataFrame({n: metrics(y[te], p, THR[n]) for n, p in clean.items()}).T
tab["thr"] = pd.Series(THR)
print("=== CLEAN TEST (thresholds chosen on val; stack = 0.5) ===")
print(tab.round(4).to_string())

def scenario(kind, mod):
    Zs = {m: Z[m][te].copy() for m in MODS}; Er = {m: Es[m][te].copy() for m in MODS}; mask = ok.copy()
    if kind == "missing":
        mask[:, MODS.index(mod)] = False; Zs[mod][:] = 0; Er[mod][:] = 0
    elif kind == "garbage":
        perm = rng.permutation(len(te)); Zs[mod] = Zs[mod][perm]; Er[mod] = Er[mod][perm]
    return Zs, Er, mask

acc, f1s = {}, {}
for kind in ("missing", "garbage"):
    for m in MODS:
        sc = all_scores(*scenario(kind, m))
        acc[f"{kind} {m}"]  = {n: metrics(y[te], p, THR[n])["acc"] for n, p in sc.items()}
        f1s[f"{kind} {m}"]  = {n: metrics(y[te], p, THR[n])["macro_f1"] for n, p in sc.items()}
acc["clean"] = {n: tab.loc[n, "acc"] for n in clean if not n.startswith("only_")}
f1s["clean"] = {n: tab.loc[n, "macro_f1"] for n in clean if not n.startswith("only_")}
A = pd.DataFrame(acc).T.loc[["clean"] + [k for k in acc if k != "clean"]]
print("\n=== ACCURACY when one sensor is missing (a = audio, r = RF) or sends garbage (rows shuffled) ===")
print(A.round(4).to_string())
print("\n=== MACRO-F1 ===")
print(pd.DataFrame(f1s).T.loc[A.index].round(4).to_string())

In [ ]:
# ============================================================
# 9. SAVE everything the demo and the report need
# ============================================================
tab.to_csv(f"{OUT_DIR}/fusion_clean_test.csv"); A.to_csv(f"{OUT_DIR}/fusion_scenarios_acc.csv")
F.drop(columns=["filename"], errors="ignore").to_csv(f"{OUT_DIR}/fusion_table.csv", index=False)
bundle = dict(mods=MODS, temperature=TEMP, thresholds=THR, vision_llr=(VLLR.tolist() if VLLR is not None else None), vision_edges=list(V_EDGES),
              stack=dict(coef=LRS["stack"].coef_.tolist(), bias=LRS["stack"].intercept_.tolist()),
              stack_drop=dict(coef=LRS["stack_drop"].coef_.tolist(), bias=LRS["stack_drop"].intercept_.tolist()))
json.dump(bundle, open(f"{OUT_DIR}/fusion_bundle.json", "w"), indent=1)
np.savez(f"{OUT_DIR}/fusion_norm.npz", **{f"mu_{m}": mu[m] for m in MODS}, **{f"sd_{m}": sd[m] for m in MODS})
if GMUS:
    torch.save([m_.state_dict() for m_ in GMUS], f"{OUT_DIR}/gmu_seeds.pt")
print("saved to", OUT_DIR, os.listdir(OUT_DIR))